# NOTEBOOK 06 — TRUSTWORTHY ML: SHAP, UNCERTAINTY & OOD DETECTION

This notebook extends the final band-gap model with three research-grade components: model explanation, uncertainty estimation, and an applicability-domain check. These are designed to answer not only **what** the model predicts, but also **why** and **when we should trust it**.


In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.models import create_model_pipeline
from src.preprocessing import FEATURE_COLUMNS, NUMERIC_FEATURES

DATA_FILE = ROOT / 'data' / 'enhanced_material_descriptors.csv'
RESULTS = ROOT / 'results'
FIGURES = ROOT / 'figures'
FIGURES.mkdir(exist_ok=True)

df = pd.read_csv(DATA_FILE)
X = df[FEATURE_COLUMNS].copy()
y = pd.to_numeric(df['target_bandgap'], errors='coerce')
valid = y.notna()
X = X.loc[valid].reset_index(drop=True)
y = y.loc[valid].reset_index(drop=True)
print(f'Materials: {len(X):,}')
print(f'Features: {len(FEATURE_COLUMNS)}')


## 1. Recreate the final evaluation split

The untouched test set remains separate from the uncertainty calibration and interpretation work. This prevents the advanced analysis from silently changing the reported test performance.


In [ ]:
y_class = (y > 0).astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y_class
)

model = create_model_pipeline()
model.fit(X_train, y_train)
test_pred = model.predict(X_test)

print(f'MAE  : {mean_absolute_error(y_test, test_pred):.4f} eV')
print(f'RMSE : {mean_squared_error(y_test, test_pred) ** 0.5:.4f} eV')
print(f'R²   : {r2_score(y_test, test_pred):.4f}')


## 2. Random-Forest uncertainty proxy

The spread of individual tree predictions is used as a **model-dispersion proxy**. It is useful for ranking confidence, but it is **not a calibrated prediction interval**.


In [ ]:
preprocessor = model.named_steps['preprocessor']
forest = model.named_steps['model']
X_test_transformed = preprocessor.transform(X_test)

tree_predictions = np.vstack([
    tree.predict(X_test_transformed) for tree in forest.estimators_
])
rf_uncertainty = tree_predictions.std(axis=0)

uncertainty_df = pd.DataFrame({
    'actual_bandgap': y_test.to_numpy(),
    'predicted_bandgap': test_pred,
    'uncertainty_proxy': rf_uncertainty,
})
uncertainty_df['absolute_error'] = (
    uncertainty_df['actual_bandgap'] - uncertainty_df['predicted_bandgap']
).abs()

print(uncertainty_df.describe())
print('\nMean absolute error by uncertainty quartile:')
uncertainty_df['uncertainty_bin'] = pd.qcut(
    uncertainty_df['uncertainty_proxy'], q=4, duplicates='drop'
)
print(uncertainty_df.groupby('uncertainty_bin', observed=True)['absolute_error'].mean())


## 3. Split-conformal prediction intervals

A calibration subset is used to estimate residual quantiles. The resulting interval is a statistical calibration layer under the usual exchangeability assumptions; it should not be interpreted as a physical uncertainty bar.


In [ ]:
X_fit, X_cal, y_fit, y_cal = train_test_split(
    X_train, y_train, test_size=0.20, random_state=123, stratify=(y_train > 0).astype(int)
)

conformal_model = create_model_pipeline()
conformal_model.fit(X_fit, y_fit)
cal_pred = conformal_model.predict(X_cal)
cal_abs_residual = np.abs(y_cal.to_numpy() - cal_pred)

alpha = 0.10
q_level = min(1.0, np.ceil((len(cal_abs_residual) + 1) * (1 - alpha)) / len(cal_abs_residual))
q = np.quantile(cal_abs_residual, q_level, method='higher')

conformal_test_pred = conformal_model.predict(X_test)
lower = conformal_test_pred - q
upper = conformal_test_pred + q
coverage = np.mean((y_test.to_numpy() >= lower) & (y_test.to_numpy() <= upper))

print(f'Calibration quantile q = {q:.4f} eV')
print(f'Nominal coverage       = {(1-alpha)*100:.1f}%')
print(f'Observed test coverage = {coverage*100:.2f}%')
print(f'Interval width         = {2*q:.4f} eV')


## 4. Applicability-domain / OOD proxy

A model can be confidently wrong when a material is chemically unlike the training data. We therefore compute a simple nearest-neighbor distance in the standardized numerical descriptor space. Larger distances indicate weaker support from nearby training examples.


In [ ]:
X_train_num = X_train[NUMERIC_FEATURES].copy()
X_test_num = X_test[NUMERIC_FEATURES].copy()

train_medians = X_train_num.median()
X_train_num = X_train_num.fillna(train_medians)
X_test_num = X_test_num.fillna(train_medians)

scaler = StandardScaler()
train_scaled = scaler.fit_transform(X_train_num)
test_scaled = scaler.transform(X_test_num)

nn = NearestNeighbors(n_neighbors=5, metric='euclidean', n_jobs=-1)
nn.fit(train_scaled)
distances, _ = nn.kneighbors(test_scaled)
ood_score = distances.mean(axis=1)

ood_df = pd.DataFrame({
    'predicted_bandgap': test_pred,
    'absolute_error': np.abs(y_test.to_numpy() - test_pred),
    'ood_distance': ood_score,
})
ood_df['ood_bin'] = pd.qcut(ood_df['ood_distance'], q=4, duplicates='drop')
print('Mean absolute error by OOD-distance quartile:')
print(ood_df.groupby('ood_bin', observed=True)['absolute_error'].mean())
print('\nImportant: this is an applicability-domain proxy, not a universal OOD detector.')


## 5. Permutation importance on the held-out test set

Permutation importance measures how much predictive performance changes when a feature is shuffled. It is a predictive interpretation, not evidence of physical causation.


In [ ]:
perm = permutation_importance(
    model,
    X_test,
    y_test,
    scoring='neg_mean_absolute_error',
    n_repeats=5,
    random_state=42,
    n_jobs=-1,
)

importance = pd.DataFrame({
    'feature': X_test.columns,
    'importance_mae': perm.importances_mean,
    'std': perm.importances_std,
}).sort_values('importance_mae', ascending=False)
display(importance)
importance.to_csv(RESULTS / 'advanced_permutation_importance.csv', index=False)


## 6. SHAP interpretation (optional but recommended)

SHAP explains individual Random-Forest predictions. Because the preprocessing pipeline expands categorical features, this section uses the transformed feature matrix and the fitted forest directly.


In [ ]:
try:
    import shap

    X_sample = X_test.sample(min(1000, len(X_test)), random_state=42)
    X_sample_transformed = preprocessor.transform(X_sample)
    feature_names = preprocessor.get_feature_names_out()

    explainer = shap.TreeExplainer(forest)
    shap_values = explainer.shap_values(X_sample_transformed)

    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    shap_importance = pd.DataFrame({
        'transformed_feature': feature_names,
        'mean_abs_shap': mean_abs_shap,
    }).sort_values('mean_abs_shap', ascending=False)
    display(shap_importance.head(25))
    shap_importance.to_csv(RESULTS / 'shap_feature_importance.csv', index=False)
    print('SHAP analysis completed.')
except ImportError:
    print('SHAP is not installed. Run: pip install shap')


## 7. Save a trustworthiness summary

The outputs from this notebook can be used by the Streamlit application later: prediction, uncertainty proxy, conformal interval, and applicability-domain score.


In [ ]:
summary = {
    'test_mae_eV': float(mean_absolute_error(y_test, test_pred)),
    'test_rmse_eV': float(mean_squared_error(y_test, test_pred) ** 0.5),
    'test_r2': float(r2_score(y_test, test_pred)),
    'conformal_alpha': float(alpha),
    'conformal_interval_half_width_eV': float(q),
    'conformal_observed_test_coverage': float(coverage),
    'notes': [
        'Random-Forest tree spread is a model-dispersion proxy.',
        'Conformal intervals are statistical, not physical uncertainty intervals.',
        'Nearest-neighbor distance is an applicability-domain/OOD proxy.',
        'Feature importance is predictive interpretation, not causal evidence.'
    ]
}
(RESULTS / 'advanced_trustworthiness_summary.json').write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
